In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

zip_path = Path(
    "/content/drive/MyDrive/confidentiality_detector/mbert_tokenized.zip"
)

print("Dataset found:", zip_path.is_file())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found: True


In [ ]:
from zipfile import ZipFile
from datasets import load_from_disk

# Extract the dataset into Colab's local storage
with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

# Load the prepared dataset
dataset_path = "/content/mbert_tokenized"
tokenized_splits = load_from_disk(dataset_path)

# Verify dataset sizes
for name, data in tokenized_splits.items():
    print(f"{name}: {len(data)} windows")


train: 34805 windows
validation: 4352 windows
test: 4349 windows


In [ ]:
from transformers import AutoModelForTokenClassification

MODEL_NAME = "google-bert/bert-base-multilingual-cased"

TARGET_LABELS = [
    "PASSWORD", "USERNAME", "EMAIL", "PHONENUMBER",
    "FIRSTNAME", "LASTNAME", "SSN",
    "CREDITCARDNUMBER", "CREDITCARDCVV",
    "IBAN", "ACCOUNTNUMBER", "STREET", "IPV4"
]

label_names = ["O"] + [
    f"{prefix}-{entity}"
    for entity in sorted(TARGET_LABELS)
    for prefix in ("B", "I")
]

label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=27,
    label2id=label2id,
    id2label=id2label
)

print("Model:", model.config.name_or_path)
print("Number of labels:", model.config.num_labels)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Model: google-bert/bert-base-multilingual-cased
Number of labels: 27


In [ ]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100
)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
batch = data_collator([
    tokenized_splits["train"][0],
    tokenized_splits["train"][1]
])

for key, value in batch.items():
    print(key, value.shape)


input_ids torch.Size([2, 64])
attention_mask torch.Size([2, 64])
labels torch.Size([2, 64])


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/confidentiality_detector/mbert_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_steps=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    report_to="none",
    seed=42,
)

print("TrainingArguments configured successfully!")


TrainingArguments configured successfully!


In [ ]:
%pip install --use-pep517 seqeval

  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=803b244e655338f5bfa427380a2c9657fb3cf8633dff8c8eb86c79d365029633
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score
from seqeval.scheme import IOB2

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_tags = []
    predicted_tags = []

    for pred_row, label_row in zip(predictions, labels):
        true_sequence = []
        predicted_sequence = []

        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                true_sequence.append("O")
                predicted_sequence.append("O")
            else:
                true_sequence.append(id2label[int(label_id)])
                predicted_sequence.append(id2label[int(pred_id)])

        true_tags.append(true_sequence)
        predicted_tags.append(predicted_sequence)

    return {
        "precision": precision_score(
            true_tags, predicted_tags, scheme=IOB2, mode="strict"
        ),
        "recall": recall_score(
            true_tags, predicted_tags, scheme=IOB2, mode="strict"
        ),
        "f1": f1_score(
            true_tags, predicted_tags, scheme=IOB2, mode="strict"
        )
    }


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits["train"],
    eval_dataset=tokenized_splits["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer initialized successfully!")
print("Training windows:", len(trainer.train_dataset))
print("Validation windows:", len(trainer.eval_dataset))


Trainer initialized successfully!
Training windows: 34805
Validation windows: 4352


In [ ]:
train_result = trainer.train()

print("Training completed!")
print(train_result.metrics)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.020208,0.005978,0.985237,0.987138,0.986187
2,0.007365,0.003279,0.990814,0.994212,0.992510


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training completed!
{'train_runtime': 1008.5003, 'train_samples_per_second': 69.023, 'train_steps_per_second': 4.315, 'total_flos': 2742230978177616.0, 'train_loss': 0.23264572836513467, 'epoch': 2.0}


In [ ]:
from pathlib import Path
from safetensors import safe_open
import transformers

checkpoint = Path(trainer.state.best_model_checkpoint)

print("Transformers version:", transformers.__version__)
print("Best checkpoint:", checkpoint)

files = list(checkpoint.glob("*.safetensors"))
print("Weight files:", len(files))

for file in files:
    with safe_open(str(file), framework="pt", device="cpu") as f:
        keys = list(f.keys())

    layernorm_keys = [
        key for key in keys
        if "LayerNorm" in key
    ]

    print("LayerNorm keys:", layernorm_keys[:6])


Transformers version: 5.20.1
Best checkpoint: /content/drive/MyDrive/confidentiality_detector/mbert_checkpoints/checkpoint-4352
Weight files: 1
LayerNorm keys: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma']


In [ ]:
from transformers import AutoModelForTokenClassification
from safetensors import safe_open
import torch

restored_model, loading_info = AutoModelForTokenClassification.from_pretrained(
    str(checkpoint),
    local_files_only=True,
    output_loading_info=True
)

with safe_open(str(files[0]), framework="pt", device="cpu") as f:
    saved_gamma = f.get_tensor("bert.embeddings.LayerNorm.gamma")

restored_weight = restored_model.bert.embeddings.LayerNorm.weight.detach().cpu()

print("LayerNorm restored correctly:", torch.equal(saved_gamma, restored_weight))
print("Missing keys:", loading_info.get("missing_keys", []))
print("Unexpected keys:", loading_info.get("unexpected_keys", []))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LayerNorm restored correctly: True
Missing keys: set()
Unexpected keys: set()
